# Barry Plant Rental Data Scraper

In [ ]:
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/152.0.0.0 Safari/537.36"
    )
}

## Data Collection

In [2]:
api_url = (
    "https://www.barryplant.com.au/api/properties/"
    "?light=1"
    "&listing_type=lease"
    "&page=1"
    "&per_page=12"
    "&property_class=residential"
    "&status=current"
)

all_properties = []
next_url = api_url
page = 1

while next_url:
    response = requests.get(
        next_url,
        headers=headers,
        timeout=20
    )
    response.raise_for_status()

    data = response.json()
    all_properties.extend(data["results"])

    next_url = data["next"]
    page += 1

    time.sleep(0.5)

print("Collected {len(all_properties)} listings.")

Collected {len(all_properties)} listings.


## Data Preparation

In [3]:
rows = []

for p in all_properties:
    location = p.get("location") or {}
    coordinates = location.get("coordinates") or [None, None]
    
    lon = coordinates[0] if len(coordinates) >=2 else None
    lat = coordinates[1] if len(coordinates) >=2 else None
    
    rows.append({
        "listing_id": p.get("id"),
        "address": p.get("address_street_display"),
        "suburb": p.get("address_suburb"),
        "postcode": p.get("address_postcode"),
        "weekly_rent": p.get("rent"),
        "bedrooms": p.get("bedrooms"),
        "bathrooms": p.get("bathrooms"),
        "carspaces": p.get("total_parking"),
        "lat": lat,
        "lon": lon,
        "url": ("https://www.barryplant.com.au" + p.get("get_absolute_url", "")),
        "scraped_date": datetime.now().date().isoformat()
    })
    
df = pd.DataFrame(rows)

print(df.shape)
df.head()

(751, 12)


,listing_id,address,suburb,postcode,weekly_rent,bedrooms,bathrooms,carspaces,lat,lon,url,scraped_date
0,209728,19B Spring Lane,Frankston,3199,870,4,2,2,-38.149573,145.122332,https://www.barryplant.com.au/rental-propertie...,2026-09-09
1,209716,10/265 Canterbury Road,Heathmont,3135,695,3,2,2,-37.831884,145.231925,https://www.barryplant.com.au/rental-propertie...,2026-09-09
2,209715,5 Devon Drive,Doncaster East,3109,600,2,1,1,-37.789821,145.157153,https://www.barryplant.com.au/rental-propertie...,2026-09-09
3,209714,82 Marigold Crescent,Gowanbrae,3043,650,3,2,1,-37.698759,144.900539,https://www.barryplant.com.au/rental-propertie...,2026-09-09
4,209712,2/18 Kalver Street,Corio,3214,520,3,2,1,-38.069202,144.363181,https://www.barryplant.com.au/rental-propertie...,2026-09-09


In [4]:
invalid_coord_mask = (
    ~df["lat"].between(-44, -34) |
    ~df["lon"].between(140, 150)
)

df.loc[invalid_coord_mask, ["lat", "lon"]] = None

print("Invalid coordinates set to missing:", invalid_coord_mask.sum())

Invalid coordinates set to missing: 1


## Property Type

In [5]:
property_types = {"house", "apartment", "townhouse", "unit", "villa", "studio", "other"}

def get_property_type(url):
    try:
        r=requests.get(url, headers=headers, timeout=20)
        r.raise_for_status()
        
        soup = BeautifulSoup(r.text, "html.parser")
        
        type_node = soup.find(string=lambda s: s and s.strip().lower() in property_types)
        
        result = type_node.strip().title() if type_node else None
        
        time.sleep(0.2)
        
        return result
    
    except requests.RequestException:
        time.sleep(0.2)
        return None

In [6]:
df["property_type"] = df["url"].apply(get_property_type)

df["property_type"].value_counts(dropna=False)

property_type
House        434
Townhouse    109
Unit          92
Apartment     84
Other         28
Studio         3
NaN            1
Name: count, dtype: int64

## Data Quality Checks

In [7]:
print("Shape:", df.shape)
print("Duplicate listing IDs:", df["listing_id"].duplicated().sum())

missing_summary = pd.DataFrame({"missing_count": df.isna().sum(),
                                "missing_pct": (df.isna().mean() * 100).round(2)})

display(missing_summary)

assert df["listing_id"].is_unique
assert df["weekly_rent"].notna().all()

valid_coords = df["lat"].notna() & df["lon"].notna()

assert df.loc[valid_coords, "lat"].between(-44, -34).all()
assert df.loc[valid_coords, "lon"].between(140, 150).all()

Shape: (751, 13)
Duplicate listing IDs: 0


,missing_count,missing_pct
listing_id,0,0.00
address,0,0.00
suburb,0,0.00
postcode,0,0.00
weekly_rent,0,0.00
bedrooms,0,0.00
bathrooms,0,0.00
carspaces,0,0.00
lat,1,0.13
lon,1,0.13


## Output

In [ ]:
output_dir = Path("../data/landing")
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(
    output_dir / "barryplant_rental_2026.csv",
    index=False
)

df.to_parquet(
    output_dir / "barryplant_rentals_2026.parquet",
    index=False
)

print("Saved CSV and parquet files.")